# Module 4 — Code Along: JSON & Pydantic Validation (the bank-accounts story, Day 2)

Yesterday we ended Day 1 with `@dataclass BankAccount` — free `__init__`, `__repr__`, `__eq__` from the six type hints. Convenient, but it has **no validation**: a negative `balance`, or `id="one"`, sails straight through and breaks something later.

**Today we fix that.** First we look at JSON — the format an account becomes the moment it leaves Python — and where stdlib `json` bites. Then we migrate the same `@dataclass BankAccount` to a **Pydantic v2 `BaseModel`** that rejects bad data *at construction*, with field-level error messages.

Same canonical account shape, same owners (Ada, Lin, Sam) as Day 1:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

These are **minimal teaching demos** on the account domain — the real `Product`/`ProductCatalog` belongs to the lab, not here.

Run top to bottom. Cells are self-contained and re-runnable.

## JSON — the parts that bite

JSON is how an account leaves Python — over HTTP, into a file. It is **stricter and poorer** than Python: lowercase `true/false/null`, no trailing commas/comments, no `NaN`/`Infinity`, string-only keys, and **no tuples, sets, or dates**. The map is simple: object↔dict, array↔list — but a Python tuple comes *back* as a list.

In [ ]:
import json

# An account leaves Python as a JSON string. tags (a list) maps to a JSON array.
account = {"id": 1, "owner": "Ada", "account_type": "savings",
           "balance": 1500.0, "is_active": True, "tags": ["primary", "online"]}

text = json.dumps(account)
print(text)   # {"id": 1, ..., "is_active": true, "tags": ["primary", "online"]}
# Note: Python True became lowercase `true` — JSON's spelling, not Python's.

back = json.loads(text)
print(back["tags"])   # ['primary', 'online'] — the list round-tripped intact

# A tuple does NOT survive as a tuple — JSON has no tuple type, so it returns a list.
with_tuple = json.dumps({"tags": ("primary", "online")})
print(json.loads(with_tuple)["tags"], type(json.loads(with_tuple)["tags"]))   # ['primary', 'online'] <class 'list'>

## stdlib `json` — the 4 functions

Two pairs, and the **`s` stands for "string"**: `loads`/`dumps` work on strings, `load`/`dump` work on open files. That one letter is the entire mental model — there is nothing else to memorise.

In [ ]:
import json, tempfile, os

account = {"id": 2, "owner": "Lin", "account_type": "checking",
           "balance": 0.0, "is_active": True, "tags": []}

# --- the STRING pair (the 's' versions) ---
text = json.dumps(account, indent=2)   # dict -> str
again = json.loads(text)               # str  -> dict
print(again["owner"])                  # Lin

# --- the FILE pair (no 's') --- write to a temp file, then read it back.
path = os.path.join(tempfile.gettempdir(), "account.json")
with open(path, "w") as fh:
    json.dump(account, fh, indent=2)   # dict -> file
with open(path) as fh:
    from_file = json.load(fh)          # file -> dict
print(from_file["id"], "loaded from", path)   # 2 loaded from .../account.json

## JSON pitfalls — what raw `json` can't serialise

Account-flavoured data isn't always JSON-clean. A `datetime` **raises a `TypeError`**; a `float('nan')` is worse — it **silently emits invalid JSON** (`NaN`) that other languages reject. Defenses: `default=str` stringifies anything (but loses the type), or let Pydantic handle it correctly.

In [ ]:
import json
from datetime import datetime

# 1) datetime: raw json has no date type, so it raises rather than guess a format.
try:
    json.dumps({"id": 1, "owner": "Ada", "created": datetime(2026, 6, 4)})
except TypeError as exc:
    print("datetime ->", exc)   # Object of type datetime is not JSON serializable

# 2) The SILENT one — float('nan') does NOT raise; it emits invalid JSON.
bad = json.dumps({"balance": float("nan")})
print("nan      ->", bad)       # {"balance": NaN}  <- not valid JSON, no error raised

# Defense: default=str stringifies the unknown value (type is lost, but it serialises).
ok = json.dumps({"created": datetime(2026, 6, 4)}, default=str)
print("default=str ->", ok)     # {"created": "2026-06-04 00:00:00"}

## Why Pydantic, not raw dicts

A raw account `dict` accepts anything: `id="one"`, `balance=-50` — and stays **silently wrong forever**, until something downstream crashes far from the cause. Pydantic checks the data at the **boundary** (construction) and tells you *which field* failed and *why*.

In [ ]:
# A raw dict happily holds garbage — nothing validates it.
raw = {"id": "one", "owner": "Ada", "balance": -50}
print(raw)   # {'id': 'one', 'owner': 'Ada', 'balance': -50} — no complaint, wrong forever

from pydantic import BaseModel, Field, ValidationError

# A tiny model with the same boundary the dict lacked.
class CheckedAccount(BaseModel):
    id: int = Field(ge=1)        # must be a positive integer
    owner: str
    balance: float = Field(ge=0) # cannot be negative

# Pydantic catches BOTH problems at once, with field-level messages.
try:
    CheckedAccount.model_validate(raw)
except ValidationError as exc:
    for err in exc.errors():
        print(err["loc"][0], "->", err["msg"])
    # id      -> Input should be a valid integer, unable to parse string as an integer
    # balance -> Input should be greater than or equal to 0

## Migrate `BankAccount`: `@dataclass` → `BaseModel`

The headline change of the day. We keep the **same six canonical fields and the same type hints** from yesterday's `@dataclass`, but subclass `BaseModel` so the account **validates on construction**. `Field(ge=...)` encodes the rules; `default_factory=list` gives each account its own `tags` (the Pydantic equivalent of yesterday's `field(default_factory=list)`).

In [ ]:
from pydantic import BaseModel, Field, ValidationError

# Yesterday this was @dataclass. Today: BaseModel — same fields, now WITH validation.
class BankAccount(BaseModel):
    id: int = Field(ge=1)
    owner: str
    account_type: str = "savings"
    balance: float = Field(default=0.0, ge=0)  # default 0.0 (like yesterday), but never negative
    is_active: bool = True
    tags: list[str] = Field(default_factory=list)   # own list per account, not shared

ada = BankAccount(id=1, owner="Ada", balance=1500.0, tags=["primary", "online"])
print(ada)   # id=1 owner='Ada' account_type='savings' balance=1500.0 is_active=True tags=[...]

# Defaults still work — a fresh account skips the optional fields, like the dataclass did.
lin = BankAccount(id=2, owner="Lin")
print(lin.account_type, lin.balance, lin.tags)   # savings 0.0 []  (its OWN empty list)

# The new power: bad data is rejected AT CONSTRUCTION, not discovered later.
try:
    BankAccount(id=3, owner="Sam", balance=-50)
except ValidationError as exc:
    print(exc.errors()[0]["loc"][0], "->", exc.errors()[0]["msg"])   # balance -> ...>= 0

## Multiple models, one resource

One class can't serve create, update, and read — the shapes differ. **`AccountBase`** holds the shared fields; **`AccountCreate`** adds the `id` the caller must supply; **`AccountUpdate`** makes everything optional (a PATCH) and `ConfigDict(extra="forbid")` rejects unknown keys (a typo'd field name); **`BankAccount`** is the full read/storage shape.

In [ ]:
from pydantic import BaseModel, Field, ConfigDict, ValidationError

class AccountBase(BaseModel):                       # fields shared by every shape
    owner: str
    account_type: str = "savings"
    balance: float = Field(ge=0)
    tags: list[str] = Field(default_factory=list)

class AccountCreate(AccountBase):                   # POST body: caller supplies the id
    id: int = Field(ge=1)

class AccountUpdate(BaseModel):                     # PATCH body: every field optional
    owner: str | None = None
    balance: float | None = Field(default=None, ge=0)
    model_config = ConfigDict(extra="forbid")       # unknown key -> error, not ignored

class BankAccount(AccountBase):                     # read / storage shape
    id: int

# Create needs an id; a partial update touches only what it names.
new = AccountCreate(id=1, owner="Ada", balance=1500.0)
print(new.id, new.owner)                            # 1 Ada
patch = AccountUpdate(balance=1600.0)
print(patch.model_dump(exclude_unset=True))         # {'balance': 1600.0} — only the field we set

# extra='forbid' catches a typo'd field name that a dict would silently swallow.
try:
    AccountUpdate(blance=1600.0)                     # 'blance' is a typo
except ValidationError as exc:
    print(exc.errors()[0]["type"], "->", exc.errors()[0]["loc"][0])   # extra_forbidden -> blance

## Coercion vs validation

Pydantic v2 is **lax by default**: a string that *looks like* a number or bool is coerced — `"1500.0"`→`float`, `"1"`→`int`, `"true"`→`bool`. This is exactly why a CSV (where every cell is a string) can feed a typed model later with no manual parsing. When you need it exact, pass `strict=True` and the coercion stops.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BankAccount(BaseModel):
    id: int = Field(ge=1)
    owner: str
    balance: float = Field(ge=0)
    is_active: bool = True

# An all-strings payload, exactly as a CSV row or query string would deliver it.
payload = {"id": "1", "owner": "Ada", "balance": "1500.0", "is_active": "true"}
acct = BankAccount.model_validate(payload)
print(type(acct.id).__name__, type(acct.balance).__name__, type(acct.is_active).__name__)
# int float bool  <- the strings were coerced to real Python types
print(acct.id, acct.balance, acct.is_active)   # 1 1500.0 True

# strict=True turns coercion OFF — the same strings are now rejected.
try:
    BankAccount.model_validate(payload, strict=True)
except ValidationError as exc:
    print("strict ->", len(exc.errors()), "errors (id/balance/is_active not coerced)")

## `@field_validator` for custom logic

Coercion handles types; some fixes need real code. The CSV we import tomorrow stores `tags` as one pipe-joined string — `"primary|online"`. A `@field_validator(..., mode="before")` runs **before** type-checking, splits the string into a list, and passes a real list straight through untouched.

In [ ]:
from pydantic import BaseModel, Field, field_validator

class BankAccount(BaseModel):
    id: int = Field(ge=1)
    owner: str
    tags: list[str] = Field(default_factory=list)

    @field_validator("tags", mode="before")     # runs BEFORE the list[str] check
    @classmethod
    def _split_pipe_tags(cls, v):
        if isinstance(v, str):                  # a CSV cell: "primary|online"
            return [t.strip() for t in v.split("|") if t.strip()]
        return v                                # already a list -> leave it alone

# Case 1: a pipe-joined string (the CSV shape) is split into a clean list.
from_csv = BankAccount(id=1, owner="Ada", tags="primary|online")
print(from_csv.tags)   # ['primary', 'online']

# Case 2: an actual list passes through unchanged — same model, both inputs work.
from_api = BankAccount(id=2, owner="Lin", tags=["primary"])
print(from_api.tags)   # ['primary']

## FastAPI + Pydantic: free `/docs`

The payoff for typing the route. A single `AccountCreate` annotation gives FastAPI a **validated request body** (422 with field-level errors on bad input), a **serialised response** via `response_model`, and an auto-generated `/docs` page — for free. We register the route in a guarded `try/except` so the cell runs headless, with no server needed (the same pattern as Day 1's FastAPI demo).

In [ ]:
from pydantic import BaseModel, Field

class AccountCreate(BaseModel):
    id: int = Field(ge=1)
    owner: str
    balance: float = Field(ge=0)

class BankAccount(BaseModel):
    id: int
    owner: str
    balance: float

# We don't run a server here — just register the route to show the shape.
try:
    from fastapi import FastAPI
    app = FastAPI()

    @app.post("/accounts", status_code=201, response_model=BankAccount)
    def create_account(payload: AccountCreate) -> BankAccount:
        # FastAPI already validated `payload` against AccountCreate before we got here.
        return BankAccount(**payload.model_dump())

    paths = [r.path for r in app.routes if getattr(r, "path", "") == "/accounts"]
    print("fastapi installed — route registered:", paths)   # ['/accounts']
    print("that one type hint also populates /docs and /openapi.json automatically")
except ImportError:
    # No server, no problem: the validation we just learned is the point — FastAPI reuses it.
    payload = AccountCreate.model_validate({"id": "1", "owner": "Ada", "balance": "1500.0"})
    print("fastapi not installed — but the model still validates the body:", payload.model_dump())

## Where this goes next — Module 5

Our `BankAccount` is now a **validated Pydantic model**: bad data is caught at the boundary, CSV strings coerce cleanly, and one type hint wires it into FastAPI with free `/docs`.

**Next (Module 5)** we stop hand-building accounts in memory and **drive the server over HTTP with `requests`** — `GET`/`POST`/`PATCH` against the running API, wrapped in a small `APIClient` that returns these very models. The Pydantic shapes we built here become the typed payloads on the wire.

The lab takes this exact account-shaped pattern and applies it to a `Product` / `ProductCatalog` on the real FastAPI server.